# Chapter 22 — Classification Metrics

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(21)
n = 60000

amount = np.round(np.exp(rng.normal(3.4, 1.15, n)), 2)     # skewed, as money is
hour = rng.integers(0, 24, n)
age_days = np.clip(rng.gamma(2.0, 260, n), 1, 3000).round(0)
n_country = rng.choice([1, 2, 3], n, p=[0.88, 0.09, 0.03])
prior_chb = rng.poisson(0.06, n)

z = (-9.6
     + 0.95 * np.log1p(amount)
     + 1.60 * ((hour >= 1) & (hour <= 5))
     - 0.0032 * age_days
     + 1.45 * (n_country - 1)
     + 2.10 * prior_chb
     + rng.normal(0, 0.35, n))
fraud = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

pd.DataFrame({"Amount": amount, "Hour": hour, "AccountAgeDays": age_days,
              "CountriesUsed": n_country, "PriorChargebacks": prior_chb,
              "Fraud": fraud}).to_csv("transactions.csv", index=False)
print(f"wrote transactions.csv: {n:,} transactions, "
      f"{fraud.sum():,} fraudulent ({fraud.mean():.3%})")

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve, confusion_matrix, brier_score_loss)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
tx = pd.read_csv("transactions.csv")
X = tx.drop(columns="Fraud").values
y = tx["Fraud"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.35,
                                      random_state=0, stratify=y)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
m = make_pipeline(StandardScaler(),
                  LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:, 1]

print(f"held-out transactions: {len(yte):,}   fraudulent: {yte.sum()} "
      f"({yte.mean():.3%})")
pred = (p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
print(f"\nat the default threshold of 0.5:")
print(f"  flagged {pred.sum()}   caught {tp} of {yte.sum()}")
print(f"  accuracy {(tn + tp) / len(yte):.4f}")
print(f"  accuracy of flagging nothing at all: {1 - yte.mean():.4f}")
print(f"\nROC AUC          {roc_auc_score(yte, p):.4f}")
print(f"average precision {average_precision_score(yte, p):.4f}")
print(f"a random model    0.5000 ROC AUC, {yte.mean():.4f} avg precision")

### Block 2  (`c2.py`)

In [ ]:
# Precision at fixed capacity: the metric an operations team actually has.
m = make_pipeline(StandardScaler(),
                  LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:, 1]
order = np.argsort(-p)

print(f"{'reviewed':>9}{'caught':>8}{'precision':>11}{'recall':>9}{'lift':>8}")
for k in (50, 100, 250, 500, 1000, 2000):
    top = order[:k]
    prec = yte[top].mean()
    print(f"{k:>9}{yte[top].sum():>8}{prec:>11.1%}"
          f"{yte[top].sum()/yte.sum():>9.1%}{prec/yte.mean():>7.0f}x")

### Block 3  (`c3.py`)

In [ ]:
# ROC and PR describe the same model and disagree about how good it is.
m = make_pipeline(StandardScaler(),
                  LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:, 1]

fpr, tpr, _ = roc_curve(yte, p)
prec, rec, _ = precision_recall_curve(yte, p)

# At the point where the model catches half the fraud:
i = np.argmin(np.abs(tpr - 0.5))
j = np.argmin(np.abs(rec - 0.5))
print(f"at 50% of fraud caught:")
print(f"  false positive rate {fpr[i]:.4f}   <- looks tiny")
print(f"  precision           {prec[j]:.4f}   <- the same point, honestly")
print(f"  flagged {int(fpr[i]*(len(yte)-yte.sum())+0.5*yte.sum()):,} "
      f"of {len(yte):,} to catch {int(0.5*yte.sum())} frauds")
print(f"\nROC AUC {roc_auc_score(yte, p):.4f}  "
      f"average precision {average_precision_score(yte, p):.4f}")

### Block 4  (`c4.py`)

In [ ]:
# A score that ranks well need not be a probability you can trust.
lr = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
gb = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

for name, mdl in [("logistic", lr), ("gradient boosting", gb)]:
    p = mdl.predict_proba(Xte)[:, 1]
    print(f"{name:<20} AUC {roc_auc_score(yte, p):.4f}   "
          f"Brier {brier_score_loss(yte, p):.6f}   "
          f"mean predicted {p.mean():.5f}  actual {yte.mean():.5f}")

print(f"\ncalibration of the boosted model, by decile of predicted risk")
p = gb.predict_proba(Xte)[:, 1]
order = np.argsort(p)
print(f"{'bucket':>7}{'predicted':>12}{'actual':>10}{'n':>8}")
for b in range(10):
    idx = order[b*len(p)//10:(b+1)*len(p)//10]
    print(f"{b+1:>7}{p[idx].mean():>12.5f}{yte[idx].mean():>10.5f}{len(idx):>8}")

### Block 5  (`c5.py`)

In [ ]:
# Class weighting improves nothing about ranking and destroys calibration.
plain = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
bal = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=5000,
                                       class_weight="balanced")).fit(Xtr, ytr)

print(f"{'model':<12}{'AUC':>9}{'avg prec':>11}{'mean pred':>12}{'actual':>10}")
for name, m in [("plain", plain), ("balanced", bal)]:
    p = m.predict_proba(Xte)[:, 1]
    print(f"{name:<12}{roc_auc_score(yte, p):>9.4f}"
          f"{average_precision_score(yte, p):>11.4f}"
          f"{p.mean():>12.5f}{yte.mean():>10.5f}")

# Recalibrate the weighted model back onto the real scale.
cal = CalibratedClassifierCV(bal, method="isotonic", cv=5).fit(Xtr, ytr)
pc = cal.predict_proba(Xte)[:, 1]
print(f"{'recalibrated':<12}{roc_auc_score(yte, pc):>9.4f}"
      f"{average_precision_score(yte, pc):>11.4f}"
      f"{pc.mean():>12.5f}{yte.mean():>10.5f}")

### Block 6  (`c6.py`)

In [ ]:
# The threshold is a business decision. Price both errors and sweep.
REVIEW = 6.0          # analyst time per flagged transaction
LOSS = 240.0          # average loss on a fraud that gets through
CATCH = 0.85          # ASSUMPTION: review stops 85% of the fraud it sees

m = make_pipeline(StandardScaler(),
                  LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:, 1]

print(f"break-even precision: {REVIEW / (LOSS * CATCH):.2%}")
print(f"\n{'thresh':>8}{'flagged':>9}{'caught':>8}{'review $':>11}"
      f"{'loss saved $':>14}{'net $':>10}")
best = None
for th in (0.30, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002):
    pred = p >= th
    tp = int(((pred) & (yte == 1)).sum())
    cost = pred.sum() * REVIEW
    saved = tp * LOSS * CATCH
    net = saved - cost
    best = max(best or (net, th), (net, th))
    print(f"{th:>8.3f}{pred.sum():>9}{tp:>8}{cost:>11,.0f}"
          f"{saved:>14,.0f}{net:>10,.0f}")
print(f"\nbest net value at threshold {best[1]}: ${best[0]:,.0f}")